In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_ROOT = Path("/Users/moanason/Downloads/Data_ANA")

event_df_path = DATA_ROOT / "descriptive_processed_event_data.csv"
event_df = pd.read_csv(event_df_path)
frame_df = pd.read_csv(DATA_ROOT / "temporal_processed_data_frames.csv")
print(event_df.columns)
print(frame_df.columns)
event_df.head()

Index(['i', 'c', 'speaker', 'event', 'duration', 'speechrate', 'F0', 'RMS',
       'time_sec'],
      dtype='object')
Index(['id', 'cond', 'time', 'spk1', 'spk2', 'event', 'F0_A', 'RMS_A', 'F0_B',
       'RMS_B', 'time_sec'],
      dtype='object')


,i,c,speaker,event,duration,speechrate,F0,RMS,time_sec
0,5,0,1,Turn,11.356875,7.484453,256.353240,0.006155,1.988469
1,5,0,-1,Gap,0.371250,NaN,NaN,NaN,13.345344
2,5,0,0,Turn,9.095602,1.869035,109.146095,0.003943,13.716594
3,5,0,-1,Silence,1.011683,NaN,NaN,NaN,22.812195
4,5,0,0,Turn,2.005923,1.994095,121.429200,0.004715,23.823878


In [ ]:
# we only use Turn events for this synchrony measure
turns = (
    event_df
    .query("event == 'Turn'")
    .copy()
)

# reconstruct syllable counts from rate and duration
# (if speechrate is NaN, treat as 0 syllables)
# turns["n_syll"] = turns["speechrate"].fillna(0.0) * turns["duration"] # arbitrary
turns["n_syll"] = turns["speechrate"] * turns["duration"]
turns.loc[turns["speechrate"].isna(), "n_syll"] = 1.0 # assume 1 syllable if rate is NaN

# end time of each micro-turn
turns["end_sec"] = turns["time_sec"] + turns["duration"]

# sort chronologically within each dyad & condition
turns = turns.sort_values(["i", "c", "time_sec"]).reset_index(drop=True)

turns.head(20)

,i,c,speaker,event,duration,speechrate,F0,RMS,time_sec,n_syll,end_sec
0,5,0,1,Turn,11.356875,7.484453,256.353240,0.006155,1.988469,85.0,13.345344
1,5,0,0,Turn,9.095602,1.869035,109.146095,0.003943,13.716594,17.0,22.812195
2,5,0,0,Turn,2.005923,1.994095,121.429200,0.004715,23.823878,4.0,25.829801
3,5,0,0,Turn,2.965277,2.697893,104.359144,0.003429,26.540409,8.0,29.505686
4,5,0,0,Turn,3.927812,1.782163,93.160360,0.003293,30.848782,7.0,34.776594
5,5,0,0,Turn,1.788750,1.118099,88.345885,0.002250,35.974719,2.0,37.763469
6,5,0,1,Turn,0.759375,6.584362,335.106281,0.003729,38.657844,5.0,39.417219
7,5,0,0,Turn,1.071604,1.866362,94.854075,0.004107,40.328469,2.0,41.400072
8,5,0,0,Turn,0.618576,1.616615,106.936595,0.003624,41.836142,1.0,42.454719
9,5,0,1,Turn,3.819548,9.163388,231.852696,0.005215,42.977844,35.0,46.797392


In [ ]:
# for each dyad & condition, create macro-turn indices:
# macro_idx increases whenever the speaker changes
turns['macro_idx'] = turns.groupby(['i', 'c'])['speaker'].transform(
    lambda s: (s != s.shift()).cumsum()
).values

turns.head(20)

,i,c,speaker,event,duration,speechrate,F0,RMS,time_sec,n_syll,end_sec,macro_idx
0,5,0,1,Turn,11.356875,7.484453,256.353240,0.006155,1.988469,85.0,13.345344,1
1,5,0,0,Turn,9.095602,1.869035,109.146095,0.003943,13.716594,17.0,22.812195,2
2,5,0,0,Turn,2.005923,1.994095,121.429200,0.004715,23.823878,4.0,25.829801,2
3,5,0,0,Turn,2.965277,2.697893,104.359144,0.003429,26.540409,8.0,29.505686,2
4,5,0,0,Turn,3.927812,1.782163,93.160360,0.003293,30.848782,7.0,34.776594,2
5,5,0,0,Turn,1.788750,1.118099,88.345885,0.002250,35.974719,2.0,37.763469,2
6,5,0,1,Turn,0.759375,6.584362,335.106281,0.003729,38.657844,5.0,39.417219,3
7,5,0,0,Turn,1.071604,1.866362,94.854075,0.004107,40.328469,2.0,41.400072,4
8,5,0,0,Turn,0.618576,1.616615,106.936595,0.003624,41.836142,1.0,42.454719,4
9,5,0,1,Turn,3.819548,9.163388,231.852696,0.005215,42.977844,35.0,46.797392,5


##### synchronicity for speech rate

In [12]:
macro_turns = (
    turns
    .groupby(["i", "c", "macro_idx", "speaker"], as_index=False)
    .agg(
        start_time=("time_sec", "min"),
        end_time=("end_sec", "max"),
        D=("duration", "sum"),      # total duration (D^(1) or D^(2))
        N=("n_syll", "sum"),        # total syllables (N^(1) or N^(2))
        n_micro=("duration", "size")
    )
)

macro_turns = macro_turns.sort_values(["i", "c", "start_time"]).reset_index(drop=True)

macro_turns.head(20)

,i,c,macro_idx,speaker,start_time,end_time,D,N,n_micro
0,5,0,1,1,1.988469,13.345344,11.356875,85.0,1
1,5,0,2,0,13.716594,37.763469,19.783363,38.0,5
2,5,0,3,1,38.657844,39.417219,0.759375,5.0,1
3,5,0,4,0,40.328469,42.454719,1.690180,3.0,2
4,5,0,5,1,42.977844,48.445344,4.292048,36.0,2
5,5,0,6,0,48.968603,54.131674,5.163070,10.0,1
6,5,0,7,1,54.452844,56.646594,1.687909,6.0,2
7,5,0,8,0,56.967219,68.830344,9.019948,17.0,4
8,5,0,9,1,68.526594,69.674094,1.147500,9.0,1
9,5,0,10,0,69.825969,70.847761,1.021793,2.0,1


In [13]:
# z-transform per group
def z_by_group(df, group_cols, value_col):
    def zfun(x):
        mu = x.mean(skipna=True)
        sd = x.std(ddof=0, skipna=True)
        return (x - mu) / sd if sd > 0 else x * np.nan
    return df.groupby(group_cols)[value_col].transform(zfun)

# add dyad/cond factors
frame_df["dyad"] = frame_df["id"]
frame_df["cond_factor"] = frame_df["cond"]

# z-scored F0 / RMS for each stream within (dyad, cond)
frame_df["F0_A_z"]  = z_by_group(frame_df, ["dyad", "cond_factor"], "F0_A")
frame_df["F0_B_z"]  = z_by_group(frame_df, ["dyad", "cond_factor"], "F0_B")
frame_df["RMS_A_z"] = z_by_group(frame_df, ["dyad", "cond_factor"], "RMS_A")
frame_df["RMS_B_z"] = z_by_group(frame_df, ["dyad", "cond_factor"], "RMS_B")


In [14]:
F0_means = []
RMS_means = []

for _, row in macro_turns.iterrows():
    sess = row["i"]
    cond = row["c"]
    spk  = row["speaker"]     # 0 or 1
    t0, t1 = row["start_time"], row["end_time"]

    # frames in same session & condition & time window
    mask = (
        (frame_df["id"] == sess) &
        (frame_df["cond"] == cond) &
        (frame_df["time_sec"] >= t0) &
        (frame_df["time_sec"] <  t1)
    )

    if spk == 0:
        mask = mask & (frame_df["spk1"] == 1)
        f0_vals  = frame_df.loc[mask, "F0_A_z"]
        rms_vals = frame_df.loc[mask, "RMS_A_z"]
    else:
        mask = mask & (frame_df["spk2"] == 1)
        f0_vals  = frame_df.loc[mask, "F0_B_z"]
        rms_vals = frame_df.loc[mask, "RMS_B_z"]

    F0_means.append(f0_vals.mean() if len(f0_vals) > 0 else np.nan)
    RMS_means.append(rms_vals.mean() if len(rms_vals) > 0 else np.nan)

macro_turns["F0_z_mean"]  = F0_means
macro_turns["RMS_z_mean"] = RMS_means

macro_turns.head()


,i,c,macro_idx,speaker,start_time,end_time,D,N,n_micro,F0_z_mean,RMS_z_mean
0,5,0,1,1,1.988469,13.345344,11.356875,85.0,1,0.169011,0.475977
1,5,0,2,0,13.716594,37.763469,19.783363,38.0,5,-0.247230,-0.267115
2,5,0,3,1,38.657844,39.417219,0.759375,5.0,1,1.319329,-0.048998
3,5,0,4,0,40.328469,42.454719,1.690180,3.0,2,-0.350338,-0.175301
4,5,0,5,1,42.977844,48.445344,4.292048,36.0,2,-0.102589,0.245698


In [15]:
# shift to get next macro-turn's variables
macro_turns = macro_turns.sort_values(["i", "c", "start_time"]).reset_index(drop=True)

macro_turns["speaker_next"] = macro_turns.groupby(["i", "c"])["speaker"].shift(-1)
macro_turns["start_next"]   = macro_turns.groupby(["i", "c"])["start_time"].shift(-1)
macro_turns["end_next"]     = macro_turns.groupby(["i", "c"])["end_time"].shift(-1)
macro_turns["D_next"]       = macro_turns.groupby(["i", "c"])["D"].shift(-1)
macro_turns["N_next"]       = macro_turns.groupby(["i", "c"])["N"].shift(-1)
macro_turns["F0_next"]      = macro_turns.groupby(["i", "c"])["F0_z_mean"].shift(-1)
macro_turns["RMS_next"]     = macro_turns.groupby(["i", "c"])["RMS_z_mean"].shift(-1)
macro_turns["macro_idx_next"] = macro_turns.groupby(["i", "c"])["macro_idx"].shift(-1)

# keep valid pairs
pairs = macro_turns.dropna(subset=["N_next", "D_next", "speaker_next"]).copy()

print("same-speaker consecutive macro-turns:",
      (pairs["speaker"] == pairs["speaker_next"]).sum())

# speech-rate mismatch
# pairs["Delta_rate"] = (pairs["N_next"] - pairs["N"]) / (pairs["D"] + pairs["D_next"])
# pairs["Delta_rate"] = ((pairs["N_next"]) / (pairs["D"] + pairs["D_next"])) - ((pairs["N"]) / (pairs["D"] + pairs["D_next"]))


eps = 1e-9
alpha = 0.5 # weighting factor between load and tempo components

# load imbalance
delta_load = (pairs["N_next"] - pairs["N"]) / (pairs["D"] + pairs["D_next"])

# tempo mismatch (rate difference) with duration stabilisation
R  = pairs["N"]      / (pairs["D"]     )
R2 = pairs["N_next"] / (pairs["D_next"])
w = (4 * pairs["D"] * pairs["D_next"]) / ((pairs["D"] + pairs["D_next"]) ** 2)
delta_tempo = (R2 - R) * w

pairs["Delta_rate"] = alpha * delta_load + (1 - alpha) * delta_tempo

pairs["M_rate"]     = pairs["Delta_rate"].abs()

# F0/RMS mismatch between macro-turns (z-units)
pairs["F0_dev"]  = (pairs["F0_next"]  - pairs["F0_z_mean"])
pairs["F0_rate"]  = (pairs["F0_next"]  - pairs["F0_z_mean"]).abs()
pairs["RMS_dev"] = (pairs["RMS_next"] - pairs["RMS_z_mean"])
pairs["RMS_rate"] = (pairs["RMS_next"] - pairs["RMS_z_mean"]).abs()

# progress index within each dyad × condition
pairs["pair_idx"] = pairs.groupby(["i", "c"]).cumcount()
pairs["t_rel_idx"] = pairs["pair_idx"] / pairs.groupby(["i", "c"])["pair_idx"].transform("max") # arbitrary scaling, fixed by time in R

pairs.head(20)

same-speaker consecutive macro-turns: 0


,i,c,macro_idx,speaker,start_time,end_time,D,N,n_micro,F0_z_mean,...,RMS_next,macro_idx_next,Delta_rate,M_rate,F0_dev,F0_rate,RMS_dev,RMS_rate,pair_idx,t_rel_idx
0,5,0,1,1,1.988469,13.345344,11.356875,85.0,1,0.169011,...,-0.267115,2.0,-3.332780,3.332780,-0.416241,0.416241,-0.743092,0.743092,0,0.000000
1,5,0,2,0,13.716594,37.763469,19.783363,38.0,5,-0.247230,...,-0.048998,3.0,-0.471166,0.471166,1.566559,1.566559,0.218117,0.218117,1,0.010638
2,5,0,3,1,38.657844,39.417219,0.759375,5.0,1,1.319329,...,-0.175301,4.0,-2.465719,2.465719,-1.669667,1.669667,-0.126303,0.126303,2,0.021277
3,5,0,4,0,40.328469,42.454719,1.690180,3.0,2,-0.350338,...,0.245698,5.0,5.439046,5.439046,0.247749,0.247749,0.420999,0.420999,3,0.031915
4,5,0,5,1,42.977844,48.445344,4.292048,36.0,2,-0.102589,...,-0.463453,6.0,-4.572931,4.572931,-0.315771,0.315771,-0.709152,0.709152,4,0.042553
5,5,0,6,0,48.968603,54.131674,5.163070,10.0,1,-0.418360,...,-0.436301,7.0,0.308862,0.308862,0.580244,0.580244,0.027152,0.027152,5,0.053191
6,5,0,7,1,54.452844,56.646594,1.687909,6.0,2,0.161884,...,-0.337574,8.0,0.070145,0.070145,-0.387828,0.387828,0.098727,0.098727,6,0.063830
7,5,0,8,0,56.967219,68.830344,9.019948,17.0,4,-0.225944,...,-0.028516,9.0,0.799736,0.799736,0.348738,0.348738,0.309058,0.309058,7,0.074468
8,5,0,9,1,68.526594,69.674094,1.147500,9.0,1,0.122794,...,-0.303975,10.0,-4.546443,4.546443,-0.341139,0.341139,-0.275459,0.275459,8,0.085106
9,5,0,10,0,69.825969,70.847761,1.021793,2.0,1,-0.218345,...,0.080971,11.0,2.551040,2.551040,0.144435,0.144435,0.384946,0.384946,9,0.095745


In [16]:
sync_path = DATA_ROOT / "turn_sync_pairs_with_F0_RMS.csv"
pairs.to_csv(sync_path, index=False)
print("Saved:", sync_path)

Saved: /Users/moanason/Downloads/Data_ANA/turn_sync_pairs_with_F0_RMS.csv
